# Advanced Problems with Solutions: Python Decorators, Logging, and Stacked Decorators

This notebook contains advanced practice problems on decorators, logger decorators, metadata preservation, decorator factories, and decorator stacking order.

## Problem 1 — Metadata-Safe Logging Decorator

Write a decorator `logged` that:

- preserves the original function metadata
- logs the function name
- logs positional and keyword arguments
- returns the original result unchanged

Then apply it to a function `power(base, exponent=2)`.

In [1]:
from functools import wraps

def logged(fn):
    @wraps(fn)
    def inner(*args, **kwargs):
        print(f"Calling {fn.__name__} with args={args}, kwargs={kwargs}")
        result = fn(*args, **kwargs)
        print(f"{fn.__name__} returned {result!r}")
        return result
    return inner

@logged
def power(base, exponent=2):
    """Raises base to exponent."""
    return base ** exponent

assert power(3) == 9
assert power(2, exponent=5) == 32
assert power.__name__ == "power"
assert power.__doc__ == "Raises base to exponent."

power(4, exponent=3)

Calling power with args=(3,), kwargs={}
power returned 9
Calling power with args=(2,), kwargs={'exponent': 5}
power returned 32
Calling power with args=(4,), kwargs={'exponent': 3}
power returned 64


64

## Problem 2 — Timing Decorator with Exception Safety

Write a decorator `timed` that prints elapsed runtime even if the wrapped function raises an exception.

Use `try/finally` so timing always happens.

In [2]:
from functools import wraps
from time import perf_counter, sleep

def timed(fn):
    @wraps(fn)
    def inner(*args, **kwargs):
        start = perf_counter()
        try:
            return fn(*args, **kwargs)
        finally:
            end = perf_counter()
            print(f"{fn.__name__} ran for {end - start:.6f}s")
    return inner

@timed
def risky_divide(a, b):
    sleep(0.05)
    return a / b

assert risky_divide(10, 2) == 5

try:
    risky_divide(10, 0)
except ZeroDivisionError:
    print("Caught ZeroDivisionError as expected")

risky_divide ran for 0.050641s
risky_divide ran for 0.050867s
Caught ZeroDivisionError as expected


## Problem 3 — Stacked Decorators and Execution Order

Create two decorators:

- `before_after(label)` prints before and after a function call
- `uppercase_result` converts a string result to uppercase

Apply them in different orders and explain the output.

In [3]:
from functools import wraps

def before_after(label):
    def decorator(fn):
        @wraps(fn)
        def inner(*args, **kwargs):
            print(f"before {label}")
            result = fn(*args, **kwargs)
            print(f"after {label}")
            return result
        return inner
    return decorator

def uppercase_result(fn):
    @wraps(fn)
    def inner(*args, **kwargs):
        return fn(*args, **kwargs).upper()
    return inner

@before_after("outer")
@uppercase_result
def greet(name):
    return f"hello, {name}"

assert greet("Ada") == "HELLO, ADA"
print(greet("Ada"))

before outer
after outer
before outer
after outer
HELLO, ADA


Solution note:

`@before_after("outer")` is applied after `@uppercase_result`, so this is equivalent to:

`greet = before_after("outer")(uppercase_result(greet))`

At call time, the outer decorator runs first, then calls the uppercase-wrapped function.

## Problem 4 — Authorization vs Logging Order

Build two decorators:

- `authorize(role)` blocks users who do not have the required role
- `audit_log` logs successful function calls

Show why decorator order matters.

In [4]:
from functools import wraps

CURRENT_USER = {"name": "Simeon", "roles": {"user"}}

def authorize(required_role):
    def decorator(fn):
        @wraps(fn)
        def inner(*args, **kwargs):
            if required_role not in CURRENT_USER["roles"]:
                return "403 Forbidden"
            return fn(*args, **kwargs)
        return inner
    return decorator

def audit_log(fn):
    @wraps(fn)
    def inner(*args, **kwargs):
        result = fn(*args, **kwargs)
        print(f"AUDIT: {CURRENT_USER['name']} called {fn.__name__}")
        return result
    return inner

@audit_log
@authorize("admin")
def delete_database_logged_even_when_forbidden():
    return "database deleted"

@authorize("admin")
@audit_log
def delete_database_logged_only_when_allowed():
    return "database deleted"

print(delete_database_logged_even_when_forbidden())
print(delete_database_logged_only_when_allowed())

AUDIT: Simeon called delete_database_logged_even_when_forbidden
403 Forbidden
403 Forbidden


Solution note:

In the first function, logging is outside authorization, so the call is logged even when forbidden.

In the second function, authorization is outside logging, so unauthorized calls stop before the logging decorator runs.

## Problem 5 — Retry Decorator Factory

Write a decorator factory `retry(times, exceptions)` that retries a function when selected exceptions occur.

Requirements:

- retry only the given exception types
- preserve metadata
- re-raise the final exception if all retries fail
- count attempts correctly

In [5]:
from functools import wraps

def retry(times, exceptions):
    if times < 1:
        raise ValueError("times must be >= 1")

    def decorator(fn):
        @wraps(fn)
        def inner(*args, **kwargs):
            last_error = None
            for attempt in range(1, times + 1):
                try:
                    return fn(*args, **kwargs)
                except exceptions as exc:
                    last_error = exc
                    print(f"Attempt {attempt} failed: {exc}")
            raise last_error
        return inner
    return decorator

state = {"calls": 0}

@retry(times=3, exceptions=(ValueError,))
def unstable():
    state["calls"] += 1
    if state["calls"] < 3:
        raise ValueError("temporary failure")
    return "success"

assert unstable() == "success"
assert state["calls"] == 3
assert unstable.__name__ == "unstable"

unstable()

Attempt 1 failed: temporary failure
Attempt 2 failed: temporary failure


'success'

## Problem 6 — Decorator That Adds a Call Counter

Create a decorator `count_calls` that attaches a `.calls` attribute to the decorated function.

Each call should increment the counter.

In [6]:
from functools import wraps

def count_calls(fn):
    @wraps(fn)
    def inner(*args, **kwargs):
        inner.calls += 1
        return fn(*args, **kwargs)
    inner.calls = 0
    return inner

@count_calls
def add(a, b):
    return a + b

assert add.calls == 0
assert add(2, 3) == 5
assert add(10, 20) == 30
assert add.calls == 2

print(add.calls)

2


## Problem 7 — Composable Cache and Logger

Write a simple memoization decorator `memoize` and stack it with `logged`.

Then show the behavioral difference between:

```python
@logged
@memoize
```

and:

```python
@memoize
@logged
```

In [7]:
from functools import wraps

def logged(fn):
    @wraps(fn)
    def inner(*args, **kwargs):
        print(f"LOG: calling {fn.__name__}{args}{kwargs}")
        return fn(*args, **kwargs)
    return inner

def memoize(fn):
    cache = {}

    @wraps(fn)
    def inner(*args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        if key not in cache:
            cache[key] = fn(*args, **kwargs)
        return cache[key]

    inner.cache = cache
    return inner

@logged
@memoize
def square_a(n):
    print("computing square_a")
    return n * n

@memoize
@logged
def square_b(n):
    print("computing square_b")
    return n * n

print("square_a calls:")
square_a(5)
square_a(5)

print("\nsquare_b calls:")
square_b(5)
square_b(5)

square_a calls:
LOG: calling square_a(5,){}
computing square_a
LOG: calling square_a(5,){}

square_b calls:
LOG: calling square_b(5,){}
computing square_b


25

Solution note:

`square_a` logs every call because `logged` is outside the cache.

`square_b` logs only cache misses because `memoize` is outside `logged`.

## Problem 8 — Class-Based Decorator

Implement a class-based decorator `Trace` that prints when a function starts and finishes.

It should preserve the wrapped function metadata using `functools.update_wrapper`.

In [8]:
from functools import update_wrapper

class Trace:
    def __init__(self, fn):
        self.fn = fn
        update_wrapper(self, fn)

    def __call__(self, *args, **kwargs):
        print(f"TRACE START: {self.fn.__name__}")
        result = self.fn(*args, **kwargs)
        print(f"TRACE END: {self.fn.__name__}")
        return result

@Trace
def multiply(a, b):
    """Multiplies two numbers."""
    return a * b

assert multiply(6, 7) == 42
assert multiply.__name__ == "multiply"
assert multiply.__doc__ == "Multiplies two numbers."

multiply(3, 4)

TRACE START: multiply
TRACE END: multiply
TRACE START: multiply
TRACE END: multiply


12

## Problem 9 — Decorating Methods Correctly

Write a decorator `require_positive_amount` that can decorate instance methods.

It should assume the decorated method receives an `amount` argument and reject non-positive values.

In [9]:
from functools import wraps

def require_positive_amount(fn):
    @wraps(fn)
    def inner(self, amount, *args, **kwargs):
        if amount <= 0:
            raise ValueError("amount must be positive")
        return fn(self, amount, *args, **kwargs)
    return inner

class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance

    @require_positive_amount
    def deposit(self, amount):
        self.balance += amount
        return self.balance

account = BankAccount(100)
assert account.deposit(50) == 150

try:
    account.deposit(0)
except ValueError as exc:
    print(exc)

amount must be positive


## Problem 10 — Advanced Challenge: Decorator Pipeline for API Endpoints

Create an endpoint function decorated with:

1. `@json_response`
2. `@timed`
3. `@authorize("admin")`
4. `@audit_log`

The endpoint should return a dictionary.

Reason about which decorators run for authorized and unauthorized users.

In [10]:
from functools import wraps
from time import perf_counter
import json

CURRENT_USER = {"name": "Simeon", "roles": {"admin"}}

def json_response(fn):
    @wraps(fn)
    def inner(*args, **kwargs):
        result = fn(*args, **kwargs)
        return json.dumps(result)
    return inner

def timed(fn):
    @wraps(fn)
    def inner(*args, **kwargs):
        start = perf_counter()
        try:
            return fn(*args, **kwargs)
        finally:
            print(f"TIMED: {fn.__name__} took {perf_counter() - start:.6f}s")
    return inner

def authorize(role):
    def decorator(fn):
        @wraps(fn)
        def inner(*args, **kwargs):
            if role not in CURRENT_USER["roles"]:
                return {"error": "forbidden", "status": 403}
            return fn(*args, **kwargs)
        return inner
    return decorator

def audit_log(fn):
    @wraps(fn)
    def inner(*args, **kwargs):
        result = fn(*args, **kwargs)
        print(f"AUDIT: {CURRENT_USER['name']} called {fn.__name__}")
        return result
    return inner

@json_response
@timed
@authorize("admin")
@audit_log
def admin_dashboard():
    return {"status": 200, "data": ["users", "metrics", "settings"]}

print(admin_dashboard())

CURRENT_USER["roles"] = {"user"}
print(admin_dashboard())

AUDIT: Simeon called admin_dashboard
TIMED: admin_dashboard took 0.000145s
{"status": 200, "data": ["users", "metrics", "settings"]}
TIMED: admin_dashboard took 0.000003s
{"error": "forbidden", "status": 403}


Solution note:

The decoration is equivalent to:

`admin_dashboard = json_response(timed(authorize("admin")(audit_log(admin_dashboard))))`

At call time:

- `json_response` runs first
- then `timed`
- then `authorize`
- then `audit_log`, but only if authorization succeeds

Therefore unauthorized users still receive JSON and timing output, but they do not trigger the audit log.